# Discrete Time Markov Chains

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import normalize

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

import glob

In [ ]:
files = glob.glob('../data/raw/*.csv')
df = pd.concat([pd.read_csv(f, dtype={
    'start_station_id': 'str',
    'end_station_id': 'str',
    'member_casual': 'category',
    'rideable_type': 'category'
}) for f in files], ignore_index=True)

print(df.shape)
df.head()

In [ ]:
# 1. rebuild raw count matrix on new full dataset
all_stations_raw = pd.concat([df['start_station_id'], df['end_station_id']]).unique()
n_raw = len(all_stations_raw)
station_to_idx_raw = {s: i for i, s in enumerate(all_stations_raw)}

C_raw = np.zeros((n_raw, n_raw))
np.add.at(C_raw, (
    df['start_station_id'].map(station_to_idx_raw).values,
    df['end_station_id'].map(station_to_idx_raw).values
), 1)

# 2. find strongly connected components
n_components, labels = connected_components(csr_matrix(C_raw), directed=True, connection='strong')
print(f"Strongly connected components: {n_components}")

component_sizes = pd.Series(labels).value_counts()
print(component_sizes.head(10))
largest_component = component_sizes.idxmax()

stations_to_keep = [
    all_stations_raw[i]
    for i, label in enumerate(labels)
    if label == largest_component
]
print(f"Stations in main component: {len(stations_to_keep)}")
print(f"Stations dropped: {n_raw - len(stations_to_keep)}")

# 3. refilter and re-encode
df = df[df['start_station_id'].isin(stations_to_keep) &
        df['end_station_id'].isin(stations_to_keep)]

all_stations = pd.concat([df['start_station_id'], df['end_station_id']]).unique()
n = len(all_stations)
station_to_idx = {s: i for i, s in enumerate(all_stations)}
idx_to_station = {i: s for s, i in station_to_idx.items()}

df['start_idx'] = df['start_station_id'].map(station_to_idx)
df['end_idx']   = df['end_station_id'].map(station_to_idx)

print(f"Final: {n} stations, {len(df):,} trips")